# OMA v3-FAIR (M=4) pipeline — fig4/5/8 + eval & ablation
Attach the OMA-fair dataset, then **Save Version → Save & Run All (Commit)**.

In [ ]:
import os, shutil, glob
from pathlib import Path
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True); os.chdir(WORK)
inps = sorted(glob.glob('/kaggle/input/*'))
assert inps, 'No dataset attached! Add your M=4 OMA-fair dataset.'
DATA = inps[0]; print('Input:', DATA)
need_py = ['train_dl_v3_easy.py', 'train_dl_oma_maml.py', 'train_noris_oma.py', 'compare_oma_v3_easy.py', 'aggregate_fig5_oma.py', 'aggregate_fig8_oma.py', 'render_fair_plots.py']
need_mat = ['ISAC_RIS_OMA_channels_v3_fair.mat', 'ISAC_RIS_OMA_channels_v3_fair_N16.mat', 'ISAC_RIS_OMA_channels_v3_fair_N32.mat', 'ISAC_RIS_OMA_channels_v3_fair_N64.mat', 'ISAC_RIS_OMA_channels_v3_fair_TEST.mat', 'ISAC_OMA_channels_fair_noris.mat']
found = {}
for f in glob.glob(os.path.join(DATA,'**','*'), recursive=True):
    b = os.path.basename(f)
    if b in need_py or b in need_mat: shutil.copy(f, WORK/b); found[b]=True
print('Copied :', sorted(found))
print('MISSING:', [f for f in need_py+need_mat if f not in found] or 'none')
for d in ['fig4','fig5_oma','fig8_oma/ris','fig8_oma/no_ris','no_ris']:
    (WORK/d).mkdir(parents=True, exist_ok=True)
for agg,dst in [('aggregate_fig5_oma.py','fig5_oma'),('aggregate_fig8_oma.py','fig8_oma')]:
    if (WORK/agg).exists(): shutil.copy(WORK/agg, WORK/dst/agg)
nor = 'ISAC_OMA_channels_fair_noris.mat'
if (WORK/nor).exists(): shutil.copy(WORK/nor, WORK/'no_ris'/nor)
import torch
print('CUDA:', torch.cuda.is_available())
EPOCHS=40; BATCH=64; LR='1e-4'
print('EPOCHS',EPOCHS,'BATCH',BATCH,'LR',LR)


## fig4 — batch x lr sweep (Anshul-style: batch{32,64,128} x lr{1e-4,1e-5})
The canonical b=64,lr=1e-4 run is saved as policy_oma_fair_best.pt (used by eval/fig5/fig8).

In [ ]:
import subprocess, sys
def run(cmd):
    print('>>',' '.join(str(c) for c in cmd)); sys.stdout.flush()
    subprocess.run([str(c) for c in cmd], check=True)

LAM_S='150'  # OMA-fair: raise sensing penalty so the policy can't starve sensing
FIG4=[(b,lr) for b in (32,64,128) for lr in ('1e-4','1e-5')]
for (b,lr) in FIG4:
    out = 'policy_oma_fair_best.pt' if (b==64 and lr=='1e-4') else f'fig4/fig4_oma_b{b}_lr{lr}.pt'
    run(['python','-u','train_dl_oma_maml.py',
         '--mat','ISAC_RIS_OMA_channels_v3_fair.mat',
         '--epochs',EPOCHS,'--batch',b,'--lr',lr,'--w_c','0.7','--w_s','0.3','--lam_s',LAM_S,
         '--out',out,'--iter_log',f'fig4/fig4_oma_b{b}_lr{lr}_iters.npz'])

## fig5 — N=16,32,64

In [ ]:
for N in (16,32,64):
    run(['python','-u','train_dl_oma_maml.py',
         '--mat',f'ISAC_RIS_OMA_channels_v3_fair_N{N}.mat',
         '--epochs',EPOCHS,'--batch',BATCH,'--lr',LR,'--lam_s',LAM_S,
         '--out',f'fig5_oma/N{N}_b{BATCH}_lr{LR}.pt'])

## fig8 — P_max sweep (no-RIS + RIS)

In [ ]:
P_list=[5,10,15,20,25]   # low-SINR operating point centred on 15 dBm
for P in P_list:
    run(['python','-u','train_noris_oma.py',
         '--mat','no_ris/ISAC_OMA_channels_fair_noris.mat',
         '--epochs',60,'--batch',128,'--P_tot_dBm',P,'--lam_s',LAM_S,
         '--out',f'fig8_oma/no_ris/noris_oma_P{P}.pt'])
for P in P_list:
    run(['python','-u','train_dl_oma_maml.py',
         '--mat','ISAC_RIS_OMA_channels_v3_fair_N16.mat',
         '--epochs',EPOCHS,'--batch',BATCH,'--lr',LR,'--P_tot_dBm',P,'--lam_s',LAM_S,
         '--out',f'fig8_oma/ris/ris_oma_P{P}_b{BATCH}_lr{LR}.pt'])

## eval + ablation -> eval_oma_fair/ (summary.csv x2 + plots)

In [ ]:
run(['python','-u','compare_oma_v3_easy.py',
     '--mat','ISAC_RIS_OMA_channels_v3_fair_TEST.mat',
     '--out-dir','eval_oma_fair','--ckpt_maml','policy_oma_fair_best.pt'])

## aggregate -> fig5_oma_results.csv, fig8_oma_results.csv

In [ ]:
run(['python','fig5_oma/aggregate_fig5_oma.py','--batch',BATCH,'--lr',LR,'--n8_ckpt','policy_oma_fair_best.pt'])
run(['python','fig8_oma/aggregate_fig8_oma.py','--batch',BATCH,'--lr',LR])

## render paper figures -> plots/ (fig4/5/8 Anshul-style + loss/rate grids)

In [ ]:
run(['python','render_fair_plots.py','--scheme','oma','--root','.'])

## bundle (safe zip in /tmp, never self-zips)

In [ ]:
import shutil, os, glob
WORK='/kaggle/working'; stage='/tmp/of_stage'
shutil.rmtree(stage,ignore_errors=True); os.makedirs(stage)
for d in ['fig4','fig5_oma','fig8_oma','eval_oma_fair','plots']:
    s=os.path.join(WORK,d)
    if os.path.isdir(s): shutil.copytree(s, os.path.join(stage,d))
for f in glob.glob(WORK+'/*.pt')+glob.glob(WORK+'/*.npz'): shutil.copy(f,stage)
shutil.make_archive('/tmp/oma_fair_results','zip',stage)
shutil.move('/tmp/oma_fair_results.zip', os.path.join(WORK,'oma_fair_results.zip'))
print('done', round(os.path.getsize(WORK+'/oma_fair_results.zip')/1e6,1),'MB')
for f in ['eval_oma_fair/summary.csv','eval_oma_fair/ablation/summary.csv',
          'fig5_oma/fig5_oma_results.csv','fig8_oma/fig8_oma_results.csv']:
    print(f,'OK' if os.path.exists(os.path.join(WORK,f)) else 'MISSING')